In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
ratings = pd.read_csv(
    "../data/raw/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies = pd.read_csv(
    "../data/raw/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"]
)

users = pd.read_csv(
    "../data/raw/users.dat",
    sep="::",
    engine="python",
    names=["userId", "gender", "age", "occupation", "zip-code"]
)

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Users:", users.shape)

Ratings: (1000209, 4)
Movies: (3883, 3)
Users: (6040, 5)


In [3]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   userId     1000209 non-null  int64
 1   movieId    1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [4]:
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)
ratings["rating"] = ratings["rating"].astype(float)

movies["movieId"] = movies["movieId"].astype(int)

In [5]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

In [6]:
ratings.duplicated().sum()

np.int64(0)

In [7]:
ratings = ratings.drop_duplicates()

In [8]:
print("Ratings after removing duplicates:", len(ratings))

Ratings after removing duplicates: 1000209


In [9]:
print("Minimum rating:", ratings["rating"].min())
print("Maximum rating:", ratings["rating"].max())

Minimum rating: 1.0
Maximum rating: 5.0


In [10]:
invalid_ratings = ratings[
    (ratings["rating"] < 1) |
    (ratings["rating"] > 5)
]

print("Invalid ratings:", len(invalid_ratings))

Invalid ratings: 0


In [11]:
ratings = ratings.drop(columns=["timestamp"])

In [12]:
ratings.head()

,userId,movieId,rating
0,1,1193,5.0
1,1,661,3.0
2,1,914,3.0
3,1,3408,4.0
4,1,2355,5.0


In [13]:
train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

In [14]:
print("Training ratings:", train_ratings.shape)
print("Testing ratings:", test_ratings.shape)

Training ratings: (800167, 3)
Testing ratings: (200042, 3)


In [15]:
user_movie_matrix = train_ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

In [16]:
user_movie_matrix.shape

(6040, 3683)

In [17]:
user_movie_matrix.iloc[:5, :10]

movieId,1,2,3,4,5,6,7,8,9,10
userId,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
user_movie_matrix.fillna(0)

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [20]:
movies["genres"] = movies["genres"].fillna("")

In [21]:
movies["genres_clean"] = movies["genres"].str.replace(
    "|",
    " ",
    regex=False
)

In [22]:
movies[["title", "genres", "genres_clean"]].head()

,title,genres,genres_clean
0,Toy Story (1995),Animation|Children's|Comedy,Animation Children's Comedy
1,Jumanji (1995),Adventure|Children's|Fantasy,Adventure Children's Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama,Comedy Drama
4,Father of the Bride Part II (1995),Comedy,Comedy


In [23]:
movies["content"] = (
    movies["title"].fillna("") + " " +
    movies["genres_clean"]
)

In [24]:
movies[["title", "genres", "content"]].head()

,title,genres,content
0,Toy Story (1995),Animation|Children's|Comedy,Toy Story (1995) Animation Children's Comedy
1,Jumanji (1995),Adventure|Children's|Fantasy,Jumanji (1995) Adventure Children's Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama,Waiting to Exhale (1995) Comedy Drama
4,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy


In [25]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [26]:
tfidf_matrix = tfidf.fit_transform(
    movies["content"]
)

In [27]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (3883, 4369)


In [28]:
movie_similarity = cosine_similarity(tfidf_matrix)

In [29]:
movie_similarity.shape

(3883, 3883)

In [30]:
movie_indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

In [31]:
movie_indices["Toy Story (1995)"]

np.int64(0)

In [32]:
def similar_movies(movie_title, n=10):
    
    if movie_title not in movie_indices:
        return "Movie not found"
    
    idx = movie_indices[movie_title]
    
    similarity_scores = list(
        enumerate(movie_similarity[idx])
    )
    
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    similarity_scores = similarity_scores[1:n+1]
    
    movie_indexes = [
        i[0] for i in similarity_scores
    ]
    
    return movies.iloc[movie_indexes][
        ["movieId", "title", "genres"]
    ]

In [33]:
similar_movies("Toy Story (1995)", 10)

,movieId,title,genres
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy
3331,3400,We're Back! A Dinosaur's Story (1993),Animation|Children's
2892,2961,"Story of Us, The (1999)",Comedy|Drama
2039,2108,L.A. Story (1991),Comedy|Romance
12,13,Balto (1995),Animation|Children's
2205,2274,Lilian's Story (1995),Drama
292,295,"Pyromaniac's Love Story, A (1995)",Comedy|Romance
236,239,"Goofy Movie, A (1995)",Animation|Children's|Comedy|Romance
47,48,Pocahontas (1995),Animation|Children's|Musical|Romance
2490,2559,"King and I, The (1999)",Animation|Children's


In [34]:
os.makedirs("../data/processed", exist_ok=True)

In [35]:
train_ratings.to_csv(
    "../data/processed/train_ratings.csv",
    index=False
)

test_ratings.to_csv(
    "../data/processed/test_ratings.csv",
    index=False
)

In [36]:
movies.to_csv(
    "../data/processed/movies_processed.csv",
    index=False
)

In [37]:
os.listdir("../data/processed")

['movies_processed.csv', 'test_ratings.csv', 'train_ratings.csv']

In [38]:
print(train_ratings.shape)
print(test_ratings.shape)
print(user_movie_matrix.shape)
print(tfidf_matrix.shape)
print(movie_similarity.shape)

(800167, 3)
(200042, 3)
(6040, 3683)
(3883, 4369)
(3883, 3883)
